# 实验六｜P4 固定点四色垃圾分拣

把一个红、黄、绿或蓝色方块放在 P4，点击“启动预览”。摄像头在 P4 区域识别到**唯一一个有效色块**后，机械臂连续5帧同色确认后执行真机抓取，到对应垃圾桶上方松开夹爪，再回到工作台拍摄位姿。无需等待稳定帧；动作期间不会重复触发。P4 空置后才能接收下一块。

所有现场参数集中在下一单元。这里只记录真实动作与截图；投放是否落入桶中由现场观察确认。停止按钮阻止后续任务；正在运动时如需紧急停止，请使用设备急停。

In [ ]:
# 现场调参区：只修改本单元即可调整摄像头、判色、P4 抓取和四桶上方位置。
import csv, math, sys, threading, time, traceback
from pathlib import Path
from datetime import datetime
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

BASE = Path('/home/jetson/Notebook/4.机械臂OpenCV视觉处理实验')
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
CAMERA_ID = 0
ROI = (240, 210, 440, 390)
MIN_AREA = 700
FRAME_DELAY = 0.04
CONFIRM_FRAMES = 5  # 必须连续5帧检测到同一颜色，才允许抓取
MOVE_T = 1.5
GRIPPER_T = 0.5

# P4 的 Z=0.278 mm 是台面参考高度；XY/Z/Pitch 保留当前实测值。
# 恢复此前完成真机夹取时使用的 Roll；抓取点与上方点保持相同角度。
P4 = (195.9, -8.2, 20.4)
P4_GRIP_ROLL_DEG = 179.0
P4_PICK_POSE = (200.2, -12.8, 45.0, 132.7, P4_GRIP_ROLL_DEG)
P4_APPROACH_POSE = (198.2, -12.8, 75.0, 132.7, P4_GRIP_ROLL_DEG)

# 坐标单位 mm，两个角度单位度；Roll 保留原始读数，运动时折算到 0～360 度。
# 黄、绿、蓝桶升高后原 Pitch 无逆解；只调整这三项 Pitch，XYZ 与 Roll 保持不变。
bin_map = {
    'red': {'bin': '有害垃圾', 'pose': (103.3, -132.5, 175.1,104.0,-945200.4)},
    'yellow': {'bin': '其他垃圾', 'pose': (219.0, 99.8, 175.6, 100.0, 137329.9)},
    'green': {'bin': '厨余垃圾', 'pose': (122.8, 114.8, 172.8, 104.0, -945200.4)},
    'blue': {'bin': '可回收垃圾', 'pose': (207.8, -141.3, 173.3, 99.5, 137329.9)},
}
HSV_RANGES = {
    'red': [((0, 80, 50), (10, 255, 255)), ((170, 80, 50), (179, 255, 255))],
    'yellow': [((18, 80, 60), (38, 255, 255))],
    'green': [((40, 60, 35), (90, 255, 255))],
    'blue': [((90, 75, 45), (135, 255, 255))],
}
BGR = {'red': (0, 0, 255), 'yellow': (0, 255, 255),
       'green': (0, 255, 0), 'blue': (255, 0, 0)}
CN = {'red': '红', 'yellow': '黄', 'green': '绿', 'blue': '蓝'}
KERNEL = np.ones((5, 5), np.uint8)
print('真机自动分拣；摄像头', CAMERA_ID, '；P4 判色区域', ROI)

## 1. 核对固定目标点与四个桶上方点

P4 抓取点与四桶位置均在上方“现场调参区”修改。P4 的 `P4_GRIP_ROLL_DEG` 已恢复此前成功夹取时的 144.5°，抓取与上方使用同一值。Roll 原始值是累计角，程序按 360° 周期折算后再发给机械臂。桶坐标是**松开夹爪的上方点**，不再下降到原 SDK 的桶底位姿。

In [ ]:
def normal_roll_deg(value):
    """将累计横滚角折算到 0～360 度。"""
    return float(value) % 360.0

print('P4 台面参考：', P4)
print('P4 抓取：', P4_PICK_POSE)
print('P4 上方：', P4_APPROACH_POSE)
for color, info in bin_map.items():
    x, y, z, pitch, roll = info['pose']
    print(f'{CN[color]}色 → {info["bin"]}：XYZ=({x:.1f}, {y:.1f}, {z:.1f}) mm，'
          f'Pitch={pitch:.1f}°，Roll 原始={roll:.1f}°，运动用={normal_roll_deg(roll):.1f}°')

## 2. HSV 实时判色

只在 P4 画面区域识别方块，避免后方其他彩色方块触发分拣。红色保留跨 H=0/179 的两段 HSV 阈值。每次只允许一个有效色块，识别到后连续5帧同色确认后执行真机动作。

In [ ]:
def detect_color(frame, colors=None, roi=None):
    """返回有效目标列表、标注画面和四色掩码。"""
    if frame is None or frame.size == 0:
        raise ValueError('输入画面为空')
    chosen = list(bin_map) if colors is None else list(colors)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    height, width = frame.shape[:2]
    x1, y1, x2, y2 = (0, 0, width, height) if roi is None else roi
    x1, y1 = max(0, int(x1)), max(0, int(y1))
    x2, y2 = min(width, int(x2)), min(height, int(y2))
    if x1 >= x2 or y1 >= y2:
        raise ValueError('ROI 越界或为空')
    spatial = np.zeros((height, width), dtype=np.uint8)
    spatial[y1:y2, x1:x2] = 255
    canvas = frame.copy()
    detections, masks = [], {}
    for color in bin_map:
        mask = np.zeros((height, width), dtype=np.uint8)
        for lower, upper in HSV_RANGES[color]:
            mask |= cv2.inRange(hsv, np.array(lower, np.uint8), np.array(upper, np.uint8))
        mask = cv2.bitwise_and(mask, spatial)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, KERNEL)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL)
        masks[color] = mask
        if color not in chosen:
            continue
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < MIN_AREA:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            target = {'color': color, 'pixel': (x + w // 2, y + h // 2),
                      'box': (x, y, w, h), 'area': float(area)}
            detections.append(target)
            cv2.rectangle(canvas, (x, y), (x + w, y + h), BGR[color], 2)
            cv2.putText(canvas, f'{color} {area:.0f}', (max(5, x), max(25, y - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, .6, BGR[color], 2)
    if roi is not None:
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (220, 220, 220), 1)
    return detections, canvas, masks

def mask_montage(masks):
    """把四色 Mask 排成一张便于截图的图。"""
    tiles = []
    for color in bin_map:
        tile = cv2.cvtColor(masks[color], cv2.COLOR_GRAY2BGR)
        cv2.putText(tile, color, (12, 28), cv2.FONT_HERSHEY_SIMPLEX,
                    .8, BGR[color], 2)
        tiles.append(tile)
    return cv2.vconcat([cv2.hconcat(tiles[:2]), cv2.hconcat(tiles[2:])])

## 3. 固定 P4 抓取与四桶上方松爪

以下单元直接调用课程机械臂 SDK。P4 抓取和上方恢复此前可夹取的 Roll=144.5°。开始运动前检查 P4 抓取、上方、回零、拍摄位姿和所选桶上方位姿的逆解。运行顺序：张开夹爪 → P4 上方 → P4 夹取 → 抬起 → 回零转运 → 对应桶上方 → 松开夹爪 → 回零 → 拍摄位姿。摄像头在运动期间持续取帧。

In [ ]:
from arm5dof_uservo import Arm5DoFUServo
from config import ARM_POSE_DICT

def pose_from_values(values):
    """将现场填写的毫米与角度转换为机械臂接口位姿。"""
    x, y, z, pitch_deg, roll_deg = values
    return ([float(x), float(y), float(z)],
            math.radians(float(pitch_deg)),
            math.radians(normal_roll_deg(roll_deg)))

def sdk_pose(name):
    """读取课程已有的回零或拍摄位姿。"""
    x, y, z, pitch, roll = ARM_POSE_DICT[name]
    return ([float(x), float(y), float(z)], pitch, roll)

def checked_pose(arm, name, pose):
    """在运动前检查指定工具位姿是否可达。"""
    xyz, pitch, roll = pose
    adjusted = arm.adjust_tool_posi(xyz)
    if not np.all(np.isfinite(adjusted)):
        raise ValueError(f'{name} 坐标非法')
    checked_pitch = arm.auto_gen_pitch(adjusted) if pitch is None else pitch
    ok, _ = arm.inverse_kinematic(adjusted, checked_pitch, roll)
    if not ok:
        raise ValueError(f'{name} 逆解不可达：{xyz}，Pitch={math.degrees(checked_pitch):.1f}°，'
                         f'Roll={math.degrees(roll):.1f}°')
    return pose

def move_to(arm, name, pose):
    """执行并等待一个真实机械臂运动。"""
    xyz, pitch, roll = pose
    print(f'机械臂：{name} → {tuple(round(v, 1) for v in xyz)}')
    result = arm.set_tool_pose(xyz, tool_pitch=pitch, tool_roll=roll,
                               T=MOVE_T, is_soft_move=True, is_wait=True)
    if result is False:
        raise RuntimeError(f'{name} 运动失败，后续动作已停止')

def move_to_capture():
    """启动预览前把随臂摄像头移回工作台拍摄位姿。"""
    try:
        arm = Arm5DoFUServo(is_init_pose=False, config_folder=str(BASE / 'config'))
    except SystemExit as exc:
        raise RuntimeError('机械臂串口初始化失败，无法进入拍摄位姿') from exc
    try:
        pose = checked_pose(arm, '拍摄位姿', sdk_pose('capture_image'))
        move_to(arm, '拍摄位姿', pose)
    finally:
        arm.uart.close()

def run_pick_place(target):
    """按识别颜色在 P4 真机夹取，并在对应固定桶上方松爪。"""
    color = target['color']
    if color not in bin_map:
        raise ValueError(f'未知颜色：{color}')
    try:
        arm = Arm5DoFUServo(is_init_pose=False, config_folder=str(BASE / 'config'))
    except SystemExit as exc:
        raise RuntimeError('机械臂串口初始化失败，请检查串口和舵机电源') from exc
    try:
        poses = {
            '抓取上方': checked_pose(arm, '抓取上方', pose_from_values(P4_APPROACH_POSE)),
            'P4 抓取点': checked_pose(arm, 'P4 抓取点', pose_from_values(P4_PICK_POSE)),
            '回零点': checked_pose(arm, '回零点', sdk_pose('home')),
            '桶上方': checked_pose(arm, '桶上方', pose_from_values(bin_map[color]['pose'])),
            '拍摄位姿': checked_pose(arm, '拍摄位姿', sdk_pose('capture_image')),
        }
        print(f'识别 {CN[color]}色 → {bin_map[color]["bin"]}；固定 P4 抓取')
        arm.gripper_open(T=GRIPPER_T, is_wait=True)
        move_to(arm, '抓取上方', poses['抓取上方'])
        move_to(arm, 'P4 抓取点', poses['P4 抓取点'])
        arm.gripper_close(T=GRIPPER_T, is_wait=True)
        move_to(arm, '抬起物块', poses['抓取上方'])
        move_to(arm, '回零转运', poses['回零点'])
        move_to(arm, '对应垃圾桶上方', poses['桶上方'])
        arm.gripper_open(T=GRIPPER_T, is_wait=True)
        move_to(arm, '回零复位', poses['回零点'])
        move_to(arm, '恢复拍摄位姿', poses['拍摄位姿'])
        return f'{CN[color]}色 → {bin_map[color]["bin"]}：动作指令完成，桶内结果待人工确认'
    finally:
        arm.uart.close()

def verify_p4_cleared(frame, color):
    """动作后检查原颜色方块是否仍在 P4。"""
    if frame is None:
        raise RuntimeError('动作后没有实时画面，无法确认 P4')
    detections, _, _ = detect_color(frame, roi=ROI)
    if any(item['color'] == color for item in detections):
        raise RuntimeError(f'P4 仍检出{CN[color]}色方块；停止自动分拣，请检查夹取')

print('真机机械臂动作函数已就绪；启动预览后才会连接并移动机械臂')

## 4. 实时预览与连续自动分拣

运行下方单元并点击“启动预览”，机械臂先返回拍摄位姿。每次只在 P4 放一个方块；画面识别到唯一有效色块且连续5帧同色后夹取，到对应桶上方松爪，再返回拍摄位姿。动作期间不触发新任务。程序复查 P4：若同色块仍在，进入 ERROR，不会反复夹取。P4 空置后才能接收下一块。点击“保存实拍截图”可留存真实画面与四色 Mask。

In [ ]:
# 重新运行界面单元时，先释放上一次的预览线程。
if 'runtime' in globals():
    old_job = runtime.get('job_thread')
    if old_job is not None and old_job.is_alive():
        raise RuntimeError('机械臂动作仍在执行；请等待完成后再运行本单元')
    old_thread = runtime.get('thread')
    if old_thread is not None and old_thread.is_alive():
        runtime['stop'].set()
        old_thread.join(timeout=3)
        if old_thread.is_alive():
            raise RuntimeError('旧预览线程仍占用摄像头；请等待旧任务结束后再运行本单元')

color_choice = widgets.Dropdown(options=[('四色自动识别', 'all'),
    ('只识别红色', 'red'), ('只识别黄色', 'yellow'),
    ('只识别绿色', 'green'), ('只识别蓝色', 'blue')],
    value='all', description='颜色')
btn_start = widgets.Button(description='启动预览', button_style='success')
btn_snapshot = widgets.Button(description='保存实拍截图', button_style='info')
btn_stop = widgets.Button(description='停止预览', button_style='danger')
state_label = widgets.HTML(value='状态：STOPPED')
image_widget = widgets.Image(format='jpeg', width=640)
mask_widget = widgets.Image(format='jpeg', width=640)
log_box = widgets.Output(layout={'border': '1px solid #ddd', 'height': '160px', 'overflow': 'auto'})
display(widgets.VBox([widgets.HBox([color_choice, btn_start, btn_snapshot, btn_stop]),
    state_label, widgets.HTML('<b>原画标注</b>'), image_widget,
    widgets.HTML('<b>四色 Mask</b>'), mask_widget, log_box]))

ACTION_CSV = BASE / 'week6_action_log.csv'
EVIDENCE_DIR = BASE / 'evidence_week6'
lock = threading.RLock()
if ACTION_CSV.exists():
    with ACTION_CSV.open(newline='', encoding='utf-8-sig') as stream:
        previous_job_count = sum(1 for _ in csv.DictReader(stream))
else:
    previous_job_count = 0
runtime = {'thread': None, 'stop': threading.Event(), 'state': 'STOPPED',
           'frame_number': 0, 'latest': None, 'last_job': None,
           'job_number': previous_job_count, 'mode': None, 'job_thread': None,
           'candidate_color': None, 'confirm_count': 0}

def ui_log(message):
    with log_box:
        print(datetime.now().strftime('%H:%M:%S'), message)

def set_state(state, detail=''):
    with lock:
        runtime['state'] = state
    state_label.value = f'<b>状态：{state}</b>　{detail}'

def append_csv(path, fields, row):
    """逐条追加记录，保留此前真实测试记录。"""
    exists = path.exists() and path.stat().st_size > 0
    with path.open('a', newline='', encoding='utf-8-sig') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def run_job(target):
    """执行一次真机抓放；摄像头持续显示，动作期间不触发新任务。"""
    error = ''
    status = '失败'
    try:
        message = run_pick_place(target)
        time.sleep(0.8)
        if runtime['stop'].is_set():
            status = '动作指令完成，预览停止，P4 未核实'
            ui_log(message + '；预览已停止，P4 未核实')
        else:
            with lock:
                current = None if runtime['latest'] is None else runtime['latest'][0].copy()
            verify_p4_cleared(current, target['color'])
            status = 'P4 已清空，桶内待人工确认'
            ui_log(message + '；P4 已清空，桶内待人工确认')
    except Exception as exc:
        error = str(exc)
        ui_log('动作失败：' + error)
        with log_box:
            print(traceback.format_exc())
    with lock:
        runtime['job_number'] += 1
        job = {'编号': runtime['job_number'], '时间': datetime.now().isoformat(timespec='seconds'),
               '识别颜色': target['color'], '目标桶': bin_map[target['color']]['bin'],
               '像素坐标': str(target['pixel']), '模式': '真机',
               '执行状态': status, '异常': error}
        runtime['last_job'] = job
        if error:
            set_state('ERROR', '动作失败：' + error)
        elif runtime['stop'].is_set():
            set_state('STOPPED', '动作已结束，摄像头已停止')
        else:
            set_state('RESET', '等待 P4 空置后接收下一块')
    append_csv(ACTION_CSV, list(job), job)

def preview_loop():
    capture = None
    try:
        capture = cv2.VideoCapture(CAMERA_ID)
        if not capture.isOpened():
            raise RuntimeError(f'摄像头 {CAMERA_ID} 打开失败，请检查设备号与占用')
        move_to_capture()
        ok, first = capture.read()
        if not ok:
            raise RuntimeError('摄像头首次取帧失败')
        set_state('READY', f'把单个彩块放在 P4，连续{CONFIRM_FRAMES}帧同色后分拣')
        while not runtime['stop'].is_set():
            ok, raw = capture.read()
            if not ok:
                raise RuntimeError('摄像头取帧失败，请检查摄像头连接或占用')
            frame = raw
            choice = color_choice.value
            colors = None if choice == 'all' else [choice]
            detections, canvas, masks = detect_color(frame, colors, ROI)
            # 每次只允许画面内出现一个有效彩块。
            all_detections = detections if choice == 'all' else detect_color(frame, None, ROI)[0]
            single = detections[0] if len(detections) == 1 and len(all_detections) == 1 else None
            target = None
            with lock:
                state = runtime['state']
                if state == 'RESET':
                    # RESET期间不累计识别帧；P4清空后才重新开放下一任务。
                    runtime['candidate_color'] = None
                    runtime['confirm_count'] = 0
                    if not all_detections:
                        set_state('READY', 'P4 已空，可放入下一块')
                elif state == 'READY':
                    if single is None:
                        # 无目标或同时有多个目标：连续性中断，清空计数。
                        runtime['candidate_color'] = None
                        runtime['confirm_count'] = 0
                    else:
                        current_color = single['color']
                        if current_color == runtime['candidate_color']:
                            runtime['confirm_count'] += 1
                        else:
                            runtime['candidate_color'] = current_color
                            runtime['confirm_count'] = 1
                        if runtime['confirm_count'] >= CONFIRM_FRAMES:
                            target = dict(single)
                            set_state('BUSY',
                                      f'{CN[current_color]}色连续{CONFIRM_FRAMES}帧确认，开始分拣')
                            runtime['candidate_color'] = None
                            runtime['confirm_count'] = 0
                else:
                    # BUSY / ERROR / STOPPED 期间不接受识别确认。
                    runtime['candidate_color'] = None
                    runtime['confirm_count'] = 0
                runtime['frame_number'] += 1
                frame_text = f'{runtime["state"]}  frame {runtime["frame_number"]}'
                if runtime['state'] == 'READY' and runtime['candidate_color']:
                    frame_text += (f'  {runtime["candidate_color"]}: '
                                   f'{runtime["confirm_count"]}/{CONFIRM_FRAMES}')
            cv2.putText(canvas, frame_text, (12, 28), cv2.FONT_HERSHEY_SIMPLEX,
                        .8, (255, 255, 255), 2)
            montage = mask_montage(masks)
            with lock:
                runtime['latest'] = (raw.copy(), canvas.copy(), montage.copy())
            encoded, jpeg = cv2.imencode('.jpg', canvas, [cv2.IMWRITE_JPEG_QUALITY, 80])
            if encoded:
                image_widget.value = jpeg.tobytes()
            encoded, jpeg = cv2.imencode('.jpg', montage, [cv2.IMWRITE_JPEG_QUALITY, 70])
            if encoded:
                mask_widget.value = jpeg.tobytes()
            if target:
                job_thread = threading.Thread(target=run_job,
                    args=(target,), daemon=True)
                with lock:
                    runtime['job_thread'] = job_thread
                job_thread.start()
            time.sleep(FRAME_DELAY)
    except Exception as exc:
        ui_log('预览失败：' + str(exc))
        set_state('ERROR', str(exc))
    finally:
        if capture is not None:
            capture.release()
        with lock:
            if runtime['state'] != 'ERROR':
                set_state('STOPPED', '摄像头已释放')

def on_start(_):
    with lock:
        if runtime['thread'] is not None and runtime['thread'].is_alive():
            return
        if runtime['job_thread'] is not None and runtime['job_thread'].is_alive():
            ui_log('上一次机械臂动作仍在执行，请等待结束')
            return
        runtime['stop'] = threading.Event()
        runtime.update(latest=None, frame_number=0, mode='真机',
                       candidate_color=None, confirm_count=0)
        runtime['thread'] = threading.Thread(target=preview_loop, daemon=True)
        runtime['thread'].start()
    ui_log(f'启动预览：{runtime["mode"]}；P4 连续{CONFIRM_FRAMES}帧同色后分拣')

def on_snapshot(_):
    with lock:
        latest = runtime['latest']
        if latest is None:
            ui_log('当前没有可保存的实拍画面')
            return
        pictures = tuple(image.copy() for image in latest)
    EVIDENCE_DIR.mkdir(exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    for label, picture in zip(('原图', '识别', '四色Mask'), pictures):
        cv2.imwrite(str(EVIDENCE_DIR / f'{stamp}_{label}.png'), picture)
    ui_log(f'已保存真实摄像头画面：{EVIDENCE_DIR / stamp}')

def on_stop(_):
    runtime['stop'].set()
    ui_log('停止请求已发出；若机械臂正在运动，须等待动作完成或使用设备急停')

btn_start.on_click(on_start)
btn_snapshot.on_click(on_snapshot)
btn_stop.on_click(on_stop)

## 5. 现场操作

1. 按顺序运行上述代码单元，点击“启动预览”。程序会将机械臂移到 `capture_image` 拍摄位姿，开始显示实时画面。
2. 每次把一个红、黄、绿或蓝色方块放在 P4。识别到唯一有效颜色后，机械臂立即夹取，到对应垃圾桶上方松开夹爪，并返回拍摄位姿。
3. 观察方块是否确实落入桶内。动作结束后让 P4 画面空置，再放下一块。若显示 ERROR，先检查画面与夹取情况，处理后重新启动预览。
4. 需要调参时，停止预览，再修改顶部“现场调参区”：P4 抓取点、上方点、四桶上方点、ROI 或 HSV 阈值。重新从参数单元顺序运行到预览单元。

真实动作记录追加到 `week6_action_log.csv`，截图保存到 `evidence_week6/`。程序不会将“运动指令完成”当作“物块成功入桶”。